# Case Técnico Operations | RevOps Pipeline

Este notebook resolve o case em três frentes:

1. Auditoria e limpeza da base recebida.
2. Geração da base corrigida `opps_corrigido.xlsx`.
3. Construção da análise executiva `analise.html`, com foco em interpretação e valor de negócio.

Ponto crítico do case: a base está em nível de produto. Portanto, a mesma oportunidade pode aparecer em várias linhas. Para análises de pipeline, conversão, ticket médio e clientes, a unidade correta é a oportunidade única, não a linha da planilha.

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
import re
from pathlib import Path

pio.templates.default = "plotly_white"

INPUT_FILE = "opps_corrupted.xlsx"
OUTPUT_XLSX = "opps_corrigido.xlsx"
REPORT_ERROS = "relatorio_erros.html"
REPORT_ANALISE = "analise.html"

df_raw = pd.read_excel(INPUT_FILE)
df = df_raw.copy()

print(f"Linhas recebidas: {len(df_raw):,}")
print(f"Oportunidades únicas recebidas: {df_raw['Opportunity_ID'].nunique():,}")

Linhas recebidas: 413
Oportunidades únicas recebidas: 261


## 1. Padronização inicial

A limpeza evita transformar valores ausentes em texto. Isso é importante porque `NaN` não deve virar `"nan"` nem ser classificado como categoria real.

In [3]:
def clean_text(series: pd.Series) -> pd.Series:
    """Remove espaços e preserva nulos como valores ausentes."""
    return (
        series.astype("string")
        .str.strip()
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )

def parse_number(value):
    """Converte números em formato brasileiro ou americano para float."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip()
    text = re.sub(r"[^\d,.\-]", "", text)

    if text in ["", "-", ".", ","]:
        return np.nan

    if "," in text and "." in text:
        if text.rfind(",") > text.rfind("."):
            text = text.replace(".", "").replace(",", ".")
        else:
            text = text.replace(",", "")
    elif "," in text:
        text = text.replace(",", ".")

    return pd.to_numeric(text, errors="coerce")

text_cols = [
    "Opportunity_ID", "Account_ID", "Account_Name", "Opportunity_Owner",
    "Opportunity_Name", "Type", "Stage", "Amount_Currency",
    "Total_Product_Amount_Currency", "Product_Name", "Lead_Source", "Lead_Office"
]

for col in text_cols:
    if col in df.columns:
        df[col] = clean_text(df[col])

for col in ["Amount", "Total_Product_Amount", "Delivery_Length_Months"]:
    if col in df.columns:
        df[col] = df[col].map(parse_number)

for col in ["Close_Date", "Created_Date"]:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], dayfirst=True, errors="coerce")

df.head()

,Opportunity_ID,Account_ID,Account_Name,Opportunity_Owner,Opportunity_Name,Type,Stage,Amount,Amount_Currency,Total_Product_Amount,Total_Product_Amount_Currency,Product_Name,Close_Date,Created_Date,Lead_Source,Lead_Office,Stage_Duration,Delivery_Length_Months
0,OPP-10001,ACC-1031,Nova Studios,Queila Ribeiro,Nova Studios - DDM - Media - Managed Media (AO...,Change Order/Upsell,Registration,3043949.95,BRL,829077.09,BRL,DDM - Media - Managed Media (AOR),2026-04-30,2025-05-19,Inbound - Current Client (DEPRECATED),"Sao Carlos, BR",130,12.0
1,OPP-10001,ACC-1031,Nova Studios,Queila Ribeiro,Nova Studios - Content - Social - AOR - 04/2026,Change Order/Upsell,Registration,3043949.95,BRL,1475924.60,BRL,Content - Social - AOR,2026-04-30,2025-05-19,Inbound - Current Client (DEPRECATED),"Sao Carlos, BR",130,12.0
2,OPP-10001,ACC-1031,Nova Studios,Queila Ribeiro,Nova Studios - DDM - Media - Creative - 04/2026,Change Order/Upsell,Registration,3043949.95,BRL,738948.26,BRL,DDM - Media - Creative,2026-04-30,2025-05-19,Inbound - Current Client (DEPRECATED),"Sao Carlos, BR",130,12.0
3,OPP-10002,ACC-1108,Jojoba Cosmetics,Wanda Borges,Jojoba Cosmetics - DDM - Media - Activation - ...,Project - Competitive,Closed Won,85000.00,BRL,85000.00,BRL,DDM - Media - Activation - Platforms Reselling,2026-01-19,2025-05-30,Referral - Partner - Google Marketing Platform,São Paulo,85,12.0
4,OPP-10003,ACC-1094,Xylitol Beverages,Lucas Almeida,Xylitol Beverages - DDM - Media - Activation -...,Project - Not Competitive,Closed Won,40000.00,BRL,40000.00,BRL,DDM - Media - Activation - Platforms Reselling,2026-01-09,2025-07-21,Referral - Internal,"Sao Paulo, BR",92,12.0


## 2. Regras oficiais do case

As regras abaixo foram extraídas do enunciado:

1. Analisar apenas os tipos `Project - Competitive`, `Project - Not Competitive` e `Change Order/Upsell`.
2. Tratar a base como nível de produto e consolidar para oportunidade única quando necessário.
3. Usar a soma de `Total_Product_Amount` como fonte da verdade quando houver divergência com `Amount`.
4. Não tratar ausência de valor como erro em estágios iniciais `Opportunity Identified` e `Qualified`.
5. Criar uma taxonomia de `Lead_Source_Category` com seis categorias.

In [4]:
valid_types = [
    "Project - Competitive",
    "Project - Not Competitive",
    "Change Order/Upsell"
]

canonical_sources = [
    "Inbound - Current Client (DEPRECATED)",
    "Inbound - Website - Media.Monks (DEPRECATED)",
    "Inbound - Marketing (DEPRECATED)",
    "Inbound - Website (DEPRECATED)",
    "Marketing - Lead Scoring/Nurturing",
    "Website - Sales Form",
    "Prospecting - Personal (DEPRECATED)",
    "Prospecting - Current Client (DEPRECATED)",
    "Referral - Internal",
    "Referral - Personal",
    "Referral - S4 Company",
    "Referral - Partner - Google Marketing Platform",
    "Referral - Partner - Google Cloud",
    "Existing Client - Account/Growth Activity",
    "Industry Event",
    "MH Lead (DEPRECATED)",
    "Don't Know/Other (DEPRECATED)"
]

canonical_offices = [
    "Sao Carlos, BR",
    "Votorantim, BR",
    "Sao Paulo, BR"
]

canonical_stages = [
    "Opportunity Identified",
    "Qualified",
    "Registration",
    "Pitching",
    "Pitched",
    "Negotiation",
    "Closed Won"
]

initial_stages = ["Opportunity Identified", "Qualified"]

## 3. Auditoria de qualidade

O objetivo desta etapa não é apenas corrigir, mas explicar o impacto de cada problema no negócio:

| Problema | Impacto na análise |
|---|---|
| Tipos fora do escopo | Misturam receitas que não fazem parte da pergunta de negócio |
| Grafias divergentes | Fragmentam categorias e distorcem rankings |
| Divergência entre `Amount` e produtos | Superestima ou subestima receita e pipeline |
| Granularidade por produto | Pode inflar contagem de oportunidades |
| Valores ausentes em estágio inicial | Não são erro, mas devem ser excluídos de análises financeiras |

In [5]:
df["Stage_original"] = df["Stage"]
df["Lead_Office_original"] = df["Lead_Office"]
df["Lead_Source_original"] = df["Lead_Source"]

stage_map = {
    "Closed Wonn": "Closed Won",
    "Clossed Won": "Closed Won",
    "Negociation": "Negotiation",
    "Pitchinng": "Pitching",
    "Pithced": "Pitched",
    "Qualifyed": "Qualified",
    "Registrationn": "Registration"
}

office_map = {
    "São Paulo": "Sao Paulo, BR",
    "sao paulo": "Sao Paulo, BR",
    "SAO PAULO": "Sao Paulo, BR",
    "SP": "Sao Paulo, BR",
    "sp": "Sao Paulo, BR",
    "Sao Paulo": "Sao Paulo, BR",
    "São Carlos": "Sao Carlos, BR",
    "Votorantim": "Votorantim, BR"
}

source_map = {
    "inbound - current client (deprecated)": "Inbound - Current Client (DEPRECATED)",
    "Inbound - Current Client (DEPRECATED)": "Inbound - Current Client (DEPRECATED)",
    "inbound - website - media.monks (DEPRECATED)": "Inbound - Website - Media.Monks (DEPRECATED)",
    "REFERRAL - INTERNAL": "Referral - Internal",
    "Refferal - Internal": "Referral - Internal",
    "Referral - Personal": "Referral - Personal",
    "Don't Know-Other": "Don't Know/Other (DEPRECATED)",
    "existing client - account/growth activity": "Existing Client - Account/Growth Activity",
    "Existing Client-Account/Growth Activity": "Existing Client - Account/Growth Activity"
}
source_map_normalized = {str(k).strip().lower(): v for k, v in source_map.items()}

def fix_source(value):
    if pd.isna(value):
        return pd.NA
    value_clean = str(value).strip()
    if value_clean in canonical_sources:
        return value_clean
    return source_map_normalized.get(value_clean.lower(), value_clean)

# Auditoria antes de filtrar escopo
df_out_scope = df[~df["Type"].isin(valid_types)].copy()

# Correções categóricas
df["Stage"] = df["Stage"].replace(stage_map)
df["Lead_Office"] = df["Lead_Office"].replace(office_map)
df["Lead_Source"] = df["Lead_Source"].apply(fix_source).astype("string")

# Base de análise no escopo
df_scope = df[df["Type"].isin(valid_types)].copy()

# Recalcular Amount no nível da oportunidade
product_sum = (
    df_scope
    .groupby("Opportunity_ID")["Total_Product_Amount"]
    .sum(min_count=1)
    .rename("Amount_Corrigido")
)

original_amount = (
    df_scope
    .groupby("Opportunity_ID")["Amount"]
    .first()
    .rename("Amount_Original")
)

df_scope = df_scope.merge(product_sum, on="Opportunity_ID", how="left")
df_scope = df_scope.merge(original_amount, on="Opportunity_ID", how="left")

mask_financial_validation = ~df_scope["Stage"].isin(initial_stages)

df_scope["Amount_Divergente"] = (
    mask_financial_validation
    & df_scope["Amount_Corrigido"].notna()
    & (
        df_scope["Amount_Original"].isna()
        | ~np.isclose(
            df_scope["Amount_Original"].fillna(0),
            df_scope["Amount_Corrigido"].fillna(0),
            atol=0.01
        )
    )
)

df_scope["Amount"] = np.where(
    df_scope["Amount_Divergente"],
    df_scope["Amount_Corrigido"],
    df_scope["Amount_Original"]
)

lead_source_taxonomy = {
    "Inbound - Website - Media.Monks (DEPRECATED)": "Inbound",
    "Inbound - Marketing (DEPRECATED)": "Inbound",
    "Inbound - Website (DEPRECATED)": "Inbound",
    "Marketing - Lead Scoring/Nurturing": "Inbound",
    "Website - Sales Form": "Inbound",
    "Prospecting - Personal (DEPRECATED)": "Outbound",
    "Referral - Internal": "Referral",
    "Referral - Personal": "Referral",
    "Referral - S4 Company": "Referral",
    "Referral - Partner - Google Marketing Platform": "Referral",
    "Referral - Partner - Google Cloud": "Referral",
    "Inbound - Current Client (DEPRECATED)": "Customer Success",
    "Existing Client - Account/Growth Activity": "Customer Success",
    "Prospecting - Current Client (DEPRECATED)": "Customer Success",
    "Industry Event": "Event",
    "MH Lead (DEPRECATED)": "Unknown",
    "Don't Know/Other (DEPRECATED)": "Unknown"
}

df_scope["Lead_Source_Category"] = (
    df_scope["Lead_Source"]
    .map(lead_source_taxonomy)
    .fillna("Unknown")
)

audit_summary = pd.DataFrame({
    "Indicador": [
        "Linhas originais",
        "Oportunidades únicas originais",
        "Linhas fora do escopo removidas",
        "Linhas mantidas após filtro de escopo",
        "Oportunidades únicas após filtro de escopo",
        "Linhas com Stage corrigido",
        "Linhas com Lead_Office corrigido",
        "Linhas com Lead_Source corrigido",
        "Linhas com divergência de Amount",
        "Oportunidades com divergência de Amount"
    ],
    "Valor": [
        len(df_raw),
        df_raw["Opportunity_ID"].nunique(),
        len(df_out_scope),
        len(df_scope),
        df_scope["Opportunity_ID"].nunique(),
        (df_scope["Stage_original"] != df_scope["Stage"]).fillna(False).sum(),
        (df_scope["Lead_Office_original"] != df_scope["Lead_Office"]).fillna(False).sum(),
        (df_scope["Lead_Source_original"] != df_scope["Lead_Source"]).fillna(False).sum(),
        df_scope["Amount_Divergente"].sum(),
        df_scope.loc[df_scope["Amount_Divergente"], "Opportunity_ID"].nunique()
    ]
})

audit_summary

,Indicador,Valor
0,Linhas originais,413
1,Oportunidades únicas originais,261
2,Linhas fora do escopo removidas,108
3,Linhas mantidas após filtro de escopo,305
4,Oportunidades únicas após filtro de escopo,203
5,Linhas com Stage corrigido,10
6,Linhas com Lead_Office corrigido,14
7,Linhas com Lead_Source corrigido,12
8,Linhas com divergência de Amount,14
9,Oportunidades com divergência de Amount,12


## 4. Consolidação por oportunidade

A partir daqui, todas as métricas de negócio usam `df_deals`, uma tabela no nível correto de análise: uma linha por oportunidade.

In [6]:
products_by_opp = (
    df_scope
    .groupby("Opportunity_ID")["Product_Name"]
    .apply(lambda s: ", ".join(sorted(set(x for x in s.dropna().astype(str) if x.lower() != "nan"))))
    .rename("Produtos_Concatenados")
)

deal_cols = [
    "Opportunity_ID", "Account_ID", "Account_Name", "Opportunity_Owner",
    "Opportunity_Name", "Type", "Stage", "Amount", "Amount_Currency",
    "Close_Date", "Created_Date", "Lead_Source", "Lead_Source_Category",
    "Lead_Office", "Stage_Duration", "Delivery_Length_Months"
]

df_deals = (
    df_scope
    .drop_duplicates(subset=["Opportunity_ID"])
    [deal_cols]
    .merge(products_by_opp, on="Opportunity_ID", how="left")
)

df_financial = df_deals[~df_deals["Stage"].isin(initial_stages)].copy()
df_won = df_financial[df_financial["Stage"] == "Closed Won"].copy()
df_open = df_financial[df_financial["Stage"] != "Closed Won"].copy()

print(f"Deals no escopo: {df_deals['Opportunity_ID'].nunique():,}")
print(f"Deals Closed Won com valor: {df_won['Opportunity_ID'].nunique():,}")
print(f"Deals em aberto com valor: {df_open['Opportunity_ID'].nunique():,}")

Deals no escopo: 203
Deals Closed Won com valor: 67
Deals em aberto com valor: 104


## 5. Métricas executivas e interpretação

As observações abaixo foram escritas para liderança de Operations. A intenção é sair da descrição do gráfico e mostrar implicação prática para o negócio.

In [7]:
def money(value):
    if pd.isna(value):
        return "n/a"
    return f"R$ {value:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")

receita_mom = (
    df_won
    .assign(Periodo=df_won["Close_Date"].dt.to_period("M").astype(str))
    .groupby("Periodo", as_index=False)["Amount"].sum()
)

lead_share = (
    df_won
    .groupby("Lead_Source_Category", as_index=False)
    .agg(Receita=("Amount", "sum"), Deals=("Opportunity_ID", "nunique"))
)
lead_share["Share_Receita_%"] = lead_share["Receita"] / lead_share["Receita"].sum() * 100
lead_share = lead_share.sort_values("Receita", ascending=False)

top_10_open = (
    df_open
    .nlargest(10, "Amount")
    [["Account_Name", "Produtos_Concatenados", "Stage", "Amount", "Close_Date"]]
)

win_rate = (
    df_deals
    .groupby("Lead_Source_Category", as_index=False)
    .agg(
        Closed_Won=("Stage", lambda s: (s == "Closed Won").sum()),
        Total=("Opportunity_ID", "nunique")
    )
)
win_rate["Win_Rate_%"] = win_rate["Closed_Won"] / win_rate["Total"] * 100
win_rate = win_rate.sort_values("Win_Rate_%", ascending=False)

ticket_type = (
    df_won
    .groupby("Type", as_index=False)
    .agg(
        Deals=("Opportunity_ID", "nunique"),
        Ticket_Medio=("Amount", "mean"),
        Receita=("Amount", "sum")
    )
    .sort_values("Ticket_Medio", ascending=False)
)

pipeline_stage = (
    df_open
    .groupby("Stage", as_index=False)
    .agg(Pipeline=("Amount", "sum"), Deals=("Opportunity_ID", "nunique"))
    .sort_values("Pipeline", ascending=False)
)

df_won["Business_Type"] = np.where(
    df_won["Type"] == "Change Order/Upsell",
    "Upsell",
    "New Business"
)

mix_time = (
    df_won
    .assign(Periodo=df_won["Close_Date"].dt.to_period("M").astype(str))
    .groupby(["Periodo", "Business_Type"], as_index=False)
    .agg(Receita=("Amount", "sum"))
)

df_won["Ciclo_Venda_Dias"] = (df_won["Close_Date"] - df_won["Created_Date"]).dt.days

sales_cycle = (
    df_won
    .groupby("Lead_Office", as_index=False)
    .agg(
        Ciclo_Medio_Dias=("Ciclo_Venda_Dias", "mean"),
        Deals=("Opportunity_ID", "nunique")
    )
    .sort_values("Ciclo_Medio_Dias")
)

top_clients = (
    df_won
    .groupby("Account_Name", as_index=False)
    .agg(Receita=("Amount", "sum"), Deals=("Opportunity_ID", "nunique"))
    .nlargest(10, "Receita")
)

snapshot_date = df_deals["Created_Date"].max()
df_open["Idade_Dias"] = (snapshot_date - df_open["Created_Date"]).dt.days

pipeline_age = df_open["Idade_Dias"].describe(percentiles=[.25, .5, .75, .9])

print("Receita Closed Won total:", money(df_won["Amount"].sum()))
print("Pipeline aberto total:", money(df_open["Amount"].sum()))
print("Data de referência para aging:", snapshot_date.date())

Receita Closed Won total: R$ 27.186.885,84
Pipeline aberto total: R$ 72.902.413,69
Data de referência para aging: 2026-04-14


## 6. Geração dos relatórios HTML

Cada visualização inclui uma interpretação curta com foco em decisão: priorização de canais, gestão de pipeline, dependência de clientes, velocidade comercial e higiene do CRM.

In [8]:
def table_html(df_table, money_cols=None, percent_cols=None, date_cols=None):
    table = df_table.copy()
    money_cols = money_cols or []
    percent_cols = percent_cols or []
    date_cols = date_cols or []

    for col in money_cols:
        if col in table.columns:
            table[col] = table[col].map(money)

    for col in percent_cols:
        if col in table.columns:
            table[col] = table[col].map(lambda x: f"{x:.1f}%")

    for col in date_cols:
        if col in table.columns:
            table[col] = pd.to_datetime(table[col], errors="coerce").dt.strftime("%d/%m/%Y")

    return table.to_html(index=False, classes="data-table", border=0)

def wrap_block(title, what_it_shows, insight, content_html):
    return f"""
    <section class="card">
        <h2>{title}</h2>
        <p class="what"><strong>O que mostra:</strong> {what_it_shows}</p>
        <div>{content_html}</div>
        <p class="insight"><strong>Insight executivo:</strong> {insight}</p>
    </section>
    """

# Figuras
fig1 = px.bar(receita_mom, x="Periodo", y="Amount", text_auto=".2s")
fig1.update_layout(yaxis_title="Receita Closed Won", xaxis_title="Mês")

fig2 = px.pie(lead_share, names="Lead_Source_Category", values="Receita", hole=0.45)

fig4 = px.bar(win_rate, x="Lead_Source_Category", y="Win_Rate_%", text_auto=".1f")
fig4.update_layout(yaxis_title="Win rate (%)", xaxis_title="Categoria de Lead Source")

fig5 = px.bar(ticket_type, x="Type", y="Ticket_Medio", text_auto=".2s")
fig5.update_layout(yaxis_title="Ticket médio", xaxis_title="Tipo de oportunidade")

fig6 = px.bar(pipeline_stage, x="Stage", y="Pipeline", text_auto=".2s")
fig6.update_layout(yaxis_title="Pipeline aberto", xaxis_title="Stage")

fig7 = px.bar(mix_time, x="Periodo", y="Receita", color="Business_Type", barmode="stack")
fig7.update_layout(yaxis_title="Receita Closed Won", xaxis_title="Mês")

fig8 = px.bar(sales_cycle, x="Lead_Office", y="Ciclo_Medio_Dias", text_auto=".1f")
fig8.update_layout(yaxis_title="Dias", xaxis_title="Escritório")

fig9 = px.bar(top_clients.sort_values("Receita"), y="Account_Name", x="Receita", orientation="h", text_auto=".2s")
fig9.update_layout(yaxis_title="Cliente", xaxis_title="Receita Closed Won")

fig10 = px.histogram(df_open, x="Idade_Dias", nbins=12)
fig10.update_layout(xaxis_title="Dias desde criação", yaxis_title="Quantidade de oportunidades")

# Insights dinâmicos
top_month = receita_mom.loc[receita_mom["Amount"].idxmax()]
top_source = lead_share.iloc[0]
best_wr = win_rate[win_rate["Total"] >= 3].iloc[0]
top_ticket = ticket_type.iloc[0]
top_stage = pipeline_stage.iloc[0]
slowest_office = sales_cycle.iloc[-1]
largest_client = top_clients.iloc[0]

analysis_blocks = []

analysis_blocks.append(wrap_block(
    "1. Receita Closed Won MoM",
    "Soma mensal de Amount apenas para oportunidades Closed Won em 2026.",
    f"O maior volume fechado aparece em {top_month['Periodo']}, com {money(top_month['Amount'])}. A leitura executiva é acompanhar se os meses seguintes mantêm ritmo semelhante ou se houve concentração pontual de fechamento.",
    fig1.to_html(full_html=False, include_plotlyjs="cdn")
))

analysis_blocks.append(wrap_block(
    "2. Participação % por Lead Source",
    "Participação de receita por categoria normalizada de origem do lead.",
    f"{top_source['Lead_Source_Category']} concentra {top_source['Share_Receita_%']:.1f}% da receita Closed Won. Isso indica forte dependência desse canal e sugere monitorar risco de concentração.",
    fig2.to_html(full_html=False, include_plotlyjs=False)
))

analysis_blocks.append(wrap_block(
    "3. Top 10 Oportunidades em Aberto",
    "Maiores oportunidades em pipeline por Amount, com cliente, produtos, estágio e fechamento esperado.",
    "A lista deve orientar o foco semanal da liderança comercial: revisar próximos passos, riscos de fechamento e responsáveis pelos maiores deals.",
    table_html(top_10_open, money_cols=["Amount"], date_cols=["Close_Date"])
))

analysis_blocks.append(wrap_block(
    "4. Win Rate por Lead Source",
    "Percentual de oportunidades Closed Won dentro da base Closed Won + Open Pipeline.",
    f"Entre categorias com amostra mínima de 3 oportunidades, {best_wr['Lead_Source_Category']} apresenta o maior win rate ({best_wr['Win_Rate_%']:.1f}%). Esse canal combina volume e qualidade e merece playbook específico.",
    fig4.to_html(full_html=False, include_plotlyjs=False)
))

analysis_blocks.append(wrap_block(
    "5. Ticket médio por Type",
    "Valor médio dos deals Closed Won por tipo de oportunidade.",
    f"{top_ticket['Type']} tem o maior ticket médio ({money(top_ticket['Ticket_Medio'])}). Isso ajuda a separar estratégia de volume de estratégia de valor.",
    fig5.to_html(full_html=False, include_plotlyjs=False)
))

analysis_blocks.append(wrap_block(
    "6. Pipeline em Aberto por Stage",
    "Soma de Amount das oportunidades em aberto por etapa do funil.",
    f"O maior volume está em {top_stage['Stage']}, com {money(top_stage['Pipeline'])}. A recomendação é investigar se esse acúmulo representa maturidade de negociação ou gargalo operacional.",
    fig6.to_html(full_html=False, include_plotlyjs=False)
))

analysis_blocks.append(wrap_block(
    "7. Mix New Business vs Upsell ao longo do tempo",
    "Composição mensal da receita fechada entre novos negócios e expansão de base.",
    "O mix ajuda a avaliar qualidade do crescimento: Upsell tende a refletir expansão em clientes atuais, enquanto New Business indica aquisição e diversificação da carteira.",
    fig7.to_html(full_html=False, include_plotlyjs=False)
))

analysis_blocks.append(wrap_block(
    "8. Ciclo de venda médio",
    "Dias entre Created_Date e Close_Date para oportunidades Closed Won, agrupado por Lead_Office.",
    f"{slowest_office['Lead_Office']} tem o ciclo médio mais longo ({slowest_office['Ciclo_Medio_Dias']:.1f} dias). Ações possíveis: revisar handoff, qualificação e critérios de avanço no funil.",
    fig8.to_html(full_html=False, include_plotlyjs=False)
))

analysis_blocks.append(wrap_block(
    "9. Top 10 clientes Closed Won YTD",
    "Clientes com maior receita fechada no ano.",
    f"O maior cliente é {largest_client['Account_Name']}, com {money(largest_client['Receita'])}. A leitura é dupla: reconhecer conta estratégica e medir risco de concentração.",
    fig9.to_html(full_html=False, include_plotlyjs=False)
))

analysis_blocks.append(wrap_block(
    "10. Idade do pipeline aberto",
    "Distribuição da idade das oportunidades em aberto, medida pela diferença entre a data mais recente da base e Created_Date.",
    f"A mediana de idade do pipeline é {pipeline_age['50%']:.0f} dias e o percentil 90 é {pipeline_age['90%']:.0f} dias. Deals acima desse patamar devem entrar em revisão de higiene do CRM.",
    fig10.to_html(full_html=False, include_plotlyjs=False)
))

css = """
<style>
body { font-family: Arial, sans-serif; background: #f3f5f7; color: #1f2937; margin: 0; padding: 32px; }
main { max-width: 1180px; margin: auto; }
h1 { color: #111827; margin-bottom: 4px; }
.subtitle { color: #4b5563; margin-bottom: 24px; }
.card { background: white; border-radius: 14px; padding: 22px; margin-bottom: 24px; box-shadow: 0 4px 14px rgba(0,0,0,0.07); }
.card h2 { margin-top: 0; color: #1f3a5f; }
.what { color: #4b5563; }
.insight { background: #fff7d6; border-left: 5px solid #f2c94c; padding: 14px; border-radius: 8px; }
.data-table { width: 100%; border-collapse: collapse; font-size: 13px; }
.data-table th { background: #eef2f7; text-align: left; padding: 8px; }
.data-table td { border-bottom: 1px solid #e5e7eb; padding: 8px; vertical-align: top; }
.audit { display: grid; grid-template-columns: repeat(2, minmax(0, 1fr)); gap: 12px; }
.metric { background: #eef6ff; padding: 12px; border-radius: 10px; }
</style>
"""

with open(REPORT_ANALISE, "w", encoding="utf-8") as file:
    file.write(f"""
    <html>
    <head>
        <meta charset="UTF-8">
        <title>Análise RevOps 2026</title>
        {css}
    </head>
    <body>
        <main>
            <h1>Análise Executiva de Pipeline Comercial 2026</h1>
            <p class="subtitle">Base corrigida, consolidada por oportunidade e interpretada para liderança de Operations.</p>
            {''.join(analysis_blocks)}
        </main>
    </body>
    </html>
    """)

print(f"Relatório gerado: {REPORT_ANALISE}")

Relatório gerado: analise.html


In [9]:
examples_out_scope = df_out_scope[["Opportunity_ID", "Type"]].head(10)

stage_examples = (
    df_scope[df_scope["Stage_original"] != df_scope["Stage"]]
    [["Opportunity_ID", "Stage_original", "Stage"]]
    .drop_duplicates()
    .head(10)
)

office_examples = (
    df_scope[df_scope["Lead_Office_original"] != df_scope["Lead_Office"]]
    [["Opportunity_ID", "Lead_Office_original", "Lead_Office"]]
    .drop_duplicates()
    .head(10)
)

source_examples = (
    df_scope[df_scope["Lead_Source_original"] != df_scope["Lead_Source"]]
    [["Opportunity_ID", "Lead_Source_original", "Lead_Source"]]
    .drop_duplicates()
    .head(10)
)

amount_examples = (
    df_scope[df_scope["Amount_Divergente"]]
    [["Opportunity_ID", "Amount_Original", "Amount_Corrigido"]]
    .drop_duplicates()
    .head(10)
)

audit_html = table_html(audit_summary)

error_blocks = []

error_blocks.append(wrap_block(
    "1. Registros fora do escopo",
    "Linhas cujo Type não pertence aos três tipos solicitados no case.",
    "A remoção evita misturar oportunidades de natureza diferente, como Retainer, Renewal ou Passthrough, com a análise de projetos e upsell.",
    table_html(examples_out_scope)
))

error_blocks.append(wrap_block(
    "2. Correção de Stage",
    "Erros de digitação que fragmentariam etapas do funil.",
    "Sem esta correção, estágios como Closed Won poderiam ser subcontados, afetando receita, win rate e funil.",
    table_html(stage_examples)
))

error_blocks.append(wrap_block(
    "3. Correção de Lead Office",
    "Variações de grafia de escritório, principalmente Sao Paulo.",
    "Sem padronização, o ciclo de venda por escritório ficaria fragmentado e prejudicaria comparação regional.",
    table_html(office_examples)
))

error_blocks.append(wrap_block(
    "4. Correção de Lead Source",
    "Variações de maiúsculas, espaços, separadores e erros de digitação em fontes de lead.",
    "A padronização garante que a taxonomia de origem represente canais reais e não ruído de preenchimento.",
    table_html(source_examples)
))

error_blocks.append(wrap_block(
    "5. Divergência entre Amount e Total_Product_Amount",
    "Oportunidades em que o valor do cabeçalho não bate com a soma dos produtos.",
    "Como a fonte da verdade é a soma dos produtos, Amount foi recalculado para evitar distorção de receita e pipeline.",
    table_html(amount_examples, money_cols=["Amount_Original", "Amount_Corrigido"])
))

with open(REPORT_ERROS, "w", encoding="utf-8") as file:
    file.write(f"""
    <html>
    <head>
        <meta charset="UTF-8">
        <title>Relatório de Erros</title>
        {css}
    </head>
    <body>
        <main>
            <h1>Relatório de Auditoria e Limpeza de Dados</h1>
            <p class="subtitle">Resumo dos problemas encontrados, impacto analítico e tratamento realizado.</p>
            <section class="card">
                <h2>Resumo executivo</h2>
                {audit_html}
            </section>
            {''.join(error_blocks)}
        </main>
    </body>
    </html>
    """)

print(f"Relatório gerado: {REPORT_ERROS}")

Relatório gerado: relatorio_erros.html


## 7. Exportação da base corrigida

A base corrigida preserva a granularidade original em nível de produto, mas adiciona campos de auditoria e análise.

Para análises executivas, use `df_deals`, pois ele representa oportunidade única.

In [10]:
export_cols = [
    "Opportunity_ID", "Account_ID", "Account_Name", "Opportunity_Owner",
    "Opportunity_Name", "Type", "Stage", "Amount", "Amount_Currency",
    "Total_Product_Amount", "Total_Product_Amount_Currency", "Product_Name",
    "Close_Date", "Created_Date", "Lead_Source", "Lead_Source_Category",
    "Lead_Office", "Stage_Duration", "Delivery_Length_Months",
    "Amount_Original", "Amount_Corrigido", "Amount_Divergente",
    "Stage_original", "Lead_Office_original", "Lead_Source_original"
]

df_scope[export_cols].to_excel(OUTPUT_XLSX, index=False)

print(f"Arquivo exportado: {OUTPUT_XLSX}")
print(f"Linhas exportadas: {len(df_scope):,}")
print(f"Oportunidades únicas exportadas: {df_scope['Opportunity_ID'].nunique():,}")

Arquivo exportado: opps_corrigido.xlsx
Linhas exportadas: 305
Oportunidades únicas exportadas: 203


## 8. Síntese para apresentação final

Sugestão de narrativa para até 8 slides:

1. Contexto: objetivo era transformar uma extração de CRM em uma análise confiável para Operations.
2. Qualidade dos dados: a base tinha 413 linhas e 261 oportunidades únicas; após filtro de escopo, ficaram 305 linhas e 203 oportunidades.
3. Correções principais: 108 linhas fora do escopo, 10 linhas com Stage corrigido, 14 com Lead Office corrigido, 12 com Lead Source corrigido e 12 oportunidades com Amount divergente.
4. Resultado comercial: Closed Won soma R$ 27,19M e pipeline aberto soma R$ 72,90M.
5. Insight 1: Customer Success concentra 64,3% da receita fechada, indicando força em expansão de base e possível dependência desse canal.
6. Insight 2: Pitched e Negotiation concentram o maior volume de pipeline aberto, exigindo governança semanal dos maiores deals.
7. Insight 3: Project - Competitive possui maior ticket médio, reforçando que novos projetos competitivos são menos frequentes, mas mais relevantes em valor.
8. Recomendação: implantar validação no CRM, taxonomia fechada para campos categóricos, rotina de reconciliação Amount vs produtos e monitor de aging do pipeline.